# 04. Graph Engineering: 계약, 예산, 실패, 승인을 그래프에 넣기

03에서 순차·병렬·분기·루프를 만들었습니다. 그 흐름만으로는 운영에 넣을 수 없습니다. Worker가 무엇을 돌려줘야 하는지, 호출을 어디서 멈출지, 실패하면 무엇을 할지, 환불을 언제 실행할지가 아직 코드에 없기 때문입니다. 이 노트북은 그 네 가지를 차례로 그래프에 넣습니다. 사례는 계속 고객 요청 T-1001입니다. "결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요."

## 학습 목표

- 이 과정의 Graph Engineering 정의를 말하고, 그 정의가 2~8절의 어느 작업에 대응하는지 짚습니다.
- Prompt, Context, Harness, Graph, Loop가 각각 무엇을 설계하는지 이 과정의 부품에 대응시킵니다.
- 역할 프롬프트와 작업 계약의 차이를 Worker 출력으로 비교합니다.
- Worker 사이의 전달 계약을 스키마로 쓰고, 스키마 통과와 근거 검증이 다르다는 것을 확인합니다.
- 요약 전달과 근거 전달의 손실을 **같은 Worker 결과**에서 비교합니다.
- 실패 종류에 따라 다음 행동을 다르게 정하고 예산(budget)으로 멈춥니다.
- 병렬 Worker의 부분 실패를 합류 노드에서 드러내고, 완료 결과를 보존한 채 실패한 작업만 다시 돌립니다.
- 승인을 실행 앞에 두고, 재개와 재시도에서 중복 실행이 어떻게 생기고 어떻게 막히는지 실행해 봅니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 층 지도 | 4 |
| 2 | Prompt: 작업 계약 | 6 |
| 3 | Context: 전달 계약과 손실 | 11 |
| 4 | Harness: 공유 예산 | 5 |
| 5 | Graph와 Loop: 실패별 다음 행동 | 9 |
| 6 | 부분 실패와 합류 | 6 |
| 7 | 승인과 멱등 | 10 |
| 8 | 완성형 조립 | 5 |
| 9 | 직접 해보기와 정리 | 4 |

## 호출 규모

이 노트북은 `Day03_new/.env`의 키로 실제 Claude 모델을 부릅니다. 키 없이 도는 경로는 없습니다. 기본 실행은 모델 호출 약 31~58회, EXA 호출 0회로 예상합니다. `직접 해보기`의 유료 셀을 켜면 모델 호출이 2~4회 늘어납니다. 모델을 부르는 셀은 호출 직전에 `announce`가 규모를 한 줄로 알려 줍니다. 그래프 구조, 합류, 게이트, 멱등처럼 모델의 판단이 필요 없는 부분은 일반 함수로 돌리고 출력에 "모델 호출 0회"를 적습니다. 모델이 할 일이 없는 곳에서 모델을 부르지 않기 위해서입니다.

In [ ]:
import inspect
import json
import re
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Annotated, TypedDict

from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, Send, interrupt
from pydantic import ValidationError

from mas_lab.contracts import (FailureKind, FinalAnswer, Proposal, WorkerResult, WorkerTask, classify_failure, join_results,
                               merge_results, next_action, plan_hash, trim_payload, validate_result)
from mas_lab.desk import Desk
from mas_lab.evaluation import evaluate_row, load_gold
from mas_lab.hybrid import ROLE_GOALS, ROLES, build_hybrid
from mas_lab.ledger import GateDecision, PaymentLedger, ResponseLost, check_refund, execute_approved
from mas_lab.runner import make_desk, run_case
from mas_lab.structures import OUTCOME_GUIDE, scope_line
from mas_lab.trace import ModelMeter, Trace, timeline
from mas_lab.view import show_graph, show_table, write_design_log
from mas_lab.workers import ROLE_TOOLS, TASK_CONTRACT, allowlist_middleware, contract_prompt, make_worker, run_worker
from rag_lab.evidence import Budget, BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

require_key()  # 키가 없으면 여기서 한국어 안내와 함께 멈춥니다.
print(key_status())
model = make_model()  # 모델 객체를 만드는 것은 호출이 아닙니다.
cases = [json.loads(line) for line in Path("data/cases.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
case = next(c for c in cases if c["id"] == "complex-double-pay")
gold = load_gold()  # 사후 자동 검사에만 씁니다. 도구, 프롬프트, 게이트는 이 값을 보지 않습니다.
print("관통 사례:", case["id"], "|", case["question"])
print("측정 장치:", Desk.__name__, Trace.__name__, ModelMeter.__name__)

## 1. 층 지도: 새로운 이름은 이전 기술을 없애지 않습니다

**문제.** Agent가 틀린 답을 냈을 때 "프롬프트를 고치자"가 첫 반응이 되기 쉽습니다. 그런데 원인이 프롬프트가 아닐 수 있습니다. 필요한 조항이 Worker에게 전달되지 않았을 수도 있고, 재시도가 예산을 다 썼을 수도 있고, 실패한 Worker가 합류에서 조용히 사라졌을 수도 있습니다. 고칠 곳을 고르려면 문제를 층으로 나눠 불러야 합니다.

이 노트북은 다섯 층을 다음처럼 정의합니다. Prompt, Context, Harness 같은 이름은 시기마다 유행처럼 새로 나왔지만, 새 이름이 앞의 일을 없애지 않습니다. 잘 쓴 프롬프트는 Graph가 있어도 여전히 필요합니다.

| 층 | 설계하는 것 | 각 층의 질문 |
|---|---|---|
| Prompt | 지시 | 이 작업에서 무엇을 어떤 조건으로 하라고 말했는가? |
| Context | 제공 정보 | 이 호출에 무엇을 넣고 무엇을 뺐는가? 뺀 것은 어디에 있는가? |
| Harness | 실행 환경 | 어떤 도구를 쓸 수 있는가? 얼마까지 쓸 수 있는가? 누가 검증하고 누가 멈추는가? |
| Graph | 작업·상태·의존관계 | 어떤 일이 어떤 일을 기다리는가? 상태는 어디에 쌓이는가? |
| Loop | 반복·재시도·종료 | 언제 다시 하고, 언제 사람에게 넘기고, 언제 끝내는가? |

**이 과정의 Graph Engineering.** 이 과정에서 Graph Engineering은 작업을 노드와 간선으로 명시하고, 노드마다 누가 판단하는지(코드·모델·사람), 무엇을 주고받는지(계약), 언제 멈추고 다시 하는지(예산·실패별 다음 행동·승인)를 그래프에 적어 넣는 일입니다. 다섯 층 지도의 Graph 층을 설계하면서 나머지 네 층(Prompt·Context·Harness·Loop)을 노드와 간선에 배치하는 작업입니다. 이 노트북의 2~8절이 바로 이것을 층별로 하나씩 만듭니다.

아래 셀은 이 과정의 실제 부품을 층에 대응시킵니다. 파일과 줄은 `inspect`로 얻으므로 코드가 바뀌면 함께 바뀝니다. 마지막 칸을 읽고, 이 부품을 지웠을 때 어떤 실패가 생길지 먼저 떠올려 보세요.

In [ ]:
def where(obj) -> str:
    # 부품이 정의된 파일과 줄을 코드에서 직접 읽습니다.
    file = Path(inspect.getsourcefile(obj)).resolve().relative_to(Path.cwd().resolve())
    return f"{file}:{inspect.getsourcelines(obj)[1]}"

layers = [("Prompt", contract_prompt, "역할 이름만 받은 Worker가 다음 단계가 쓸 수 없는 출력을 냅니다"),
          ("Context", WorkerResult, "요약만 전달되어 조건과 근거가 사라집니다"),
          ("Context", trim_payload, "바이트로 잘려 예외 문장이 사라집니다"),
          ("Harness", Budget, "호출이 끝없이 늘어나도 멈추지 않습니다"),
          ("Harness", allowlist_middleware, "Worker가 허용되지 않은 도구를 부릅니다"),
          ("Harness", check_refund, "모델의 제안이 곧 실행이 됩니다"),
          ("Graph", join_results, "실패한 Worker가 합류에서 조용히 빠집니다"),
          ("Graph", build_hybrid, "경로를 모델이 매번 새로 정하게 됩니다"),
          ("Loop", next_action, "모든 실패를 똑같이 다시 호출합니다")]
show_table([{"층": layer, "부품": part.__name__, "파일과 줄": where(part), "없으면 생기는 실패": failure}
            for layer, part, failure in layers])
print("모델 호출 0회")

## 2. Prompt: 역할 이름과 작업 계약

**문제.** "너는 장애 보상 전문가야"라고 말하면 전문가처럼 일할까요? 이 지시에는 어떤 값을 근거로 삼을지, 결과를 어떤 칸에 담을지, 자료가 없으면 어떻게 할지, 무엇을 하면 안 되는지가 없습니다. 다음 단계의 코드는 Worker의 말투가 아니라 칸을 읽습니다.

`outage` Worker에게는 가용률 조회 도구만 줍니다. 이 Worker는 기본 이용료도, SLA 크레딧 구간도 볼 수 없습니다. 그래서 아래 작업은 일부러 Worker의 권한을 넘습니다. "2026년 8월 가용률을 확인하고, 그에 따른 서비스 크레딧 금액을 알려 주세요." 볼 수 없는 값을 어떻게 다루는지가 두 지시의 차이를 드러냅니다.

**먼저 예측하세요.** 역할만 있는 지시를 받은 Worker의 출력을 다음 단계의 코드가 그대로 쓸 수 있을까요? 다음 세 가지를 적어 두세요. (1) 크레딧 금액을 지어낼까요, 모른다고 할까요? (2) 근거로 쓸 조회 번호를 답에 남길까요? (3) 코드가 이 작업의 완료 여부를 어느 값으로 판단할 수 있을까요?

In [ ]:
announce("역할 프롬프트 Worker", model_calls=4)
GOAL = "2026년 8월 가용률을 확인하고, 그에 따른 서비스 크레딧 금액을 알려 주세요."
ROLE_PROMPT = "너는 장애 보상 전문가야."
role_desk, role_meter = make_desk(case, label="role"), ModelMeter()
role_agent = create_deep_agent(model=model, tools=role_desk.tools("outage"), system_prompt=ROLE_PROMPT, name="role-outage",
                               middleware=allowlist_middleware(role_desk.budget, ROLE_TOOLS["outage"]))
role_state = role_agent.invoke({"messages": [{"role": "user", "content": GOAL}]},
                               config={"callbacks": [role_meter], "recursion_limit": 40})
role_text = str(role_state["messages"][-1].text)
print("모델 호출:", role_meter.totals()["model_calls"], "| 도구:", role_desk.budget.counts.get("tool", 0))
print(role_text)

In [ ]:
announce("작업 계약 Worker", model_calls=4)
contract_task = WorkerTask(task_id="outage", goal=GOAL, allowed_tools=sorted(ROLE_TOOLS["outage"]))
contract_desk, contract_meter = make_desk(case, label="contract"), ModelMeter()
print(contract_prompt(contract_task, contract_desk.scope))
print("-" * 40)
contract_result = run_worker(contract_task, contract_desk, model, config={"callbacks": [contract_meter]})
print("모델 호출:", contract_meter.totals()["model_calls"], "| 도구:", contract_desk.budget.counts.get("tool", 0))
print(contract_result.model_dump_json(indent=2))

In [ ]:
REF = re.compile(r"\bF-[\w-]+|\bE-[0-9a-f]{12}")
AMOUNT = re.compile(r"\d+\.\d+|\d[\d,]*%|\d{1,3}(?:,\d{3})+")

def amounts(text: str) -> set[str]:
    # 소수, 비율, 쉼표 금액만 봅니다. 날짜의 숫자는 세지 않습니다.
    return {n.replace(",", "").removesuffix("%") for n in AMOUNT.findall(text)}

def source_of_numbers(text: str, desk: Desk) -> str:
    source = {n.replace(",", "") for n in re.findall(r"\d[\d,]*(?:\.\d+)?", desk.source_text())}
    found = amounts(text)
    return f"{len(found & source)}/{len(found)}개가 조회 값에 있음. 없는 값: {sorted(found - source) or '없음'}"

# 자유 텍스트에는 status 칸이 없어서 코드는 complete로 볼 수밖에 없습니다.
role_as_result = WorkerResult(status="complete", facts=[role_text], evidence_refs=REF.findall(role_text))
contract_text = " ".join([*contract_result.facts, *contract_result.uncertainty])
known = {"역할 프롬프트": role_desk.known_refs(), "작업 계약": contract_desk.known_refs()}
show_table([{"지시": "역할 프롬프트", "출력 형태": "자유 텍스트", "status": "칸 없음", "검증 가능한 근거 수": len(set(role_as_result.evidence_refs) & known["역할 프롬프트"]),
             "불확실성": "칸 없음", "사실 수": "칸 없음", "숫자의 출처": source_of_numbers(role_text, role_desk),
             "validate_result": validate_result(role_as_result, known["역할 프롬프트"]) or "문제 없음"},
            {"지시": "작업 계약", "출력 형태": "WorkerResult", "status": contract_result.status, "검증 가능한 근거 수": len(set(contract_result.evidence_refs) & known["작업 계약"]),
             "불확실성": len(contract_result.uncertainty), "사실 수": len(contract_result.facts), "숫자의 출처": source_of_numbers(contract_text, contract_desk),
             "validate_result": validate_result(contract_result, known["작업 계약"]) or "문제 없음"}])
print("모델 호출 0회")

**관찰할 것.** 표에서 `출력 형태`, `status`, `불확실성` 칸을 먼저 읽으세요. 역할 프롬프트 Worker의 답이 사람에게는 충분히 좋아 보일 수 있습니다. 모른다는 말을 정직하게 쓰고 조회 번호까지 적었을 수도 있습니다. 그렇더라도 그 정직함은 문장 안에 있습니다. 코드는 문장을 읽어 완료인지 불확실한지 판단할 수 없습니다. 작업 계약 Worker는 같은 말을 `status`와 `uncertainty` 칸에 담습니다.

`숫자의 출처` 칸은 다르게 읽습니다. 조회 값에 없는 숫자가 있다면 그 숫자가 어디서 왔는지 출력된 문장에서 직접 찾으세요. 금액을 단정하지 않고 "예를 들어"라며 일반적인 구간을 적어 둔 것일 수도 있습니다. 사람은 예시로 읽지만 다음 단계의 코드는 구별하지 못합니다. 없는 값이 없다면 이번 실행에서는 두 지시 모두 숫자를 지어내지 않은 것입니다. 이 경우에도 계약의 효과가 없다는 뜻은 아닙니다. 계약은 평균 품질이 아니라 다음 단계가 믿고 읽을 칸을 정합니다.

`validate_result` 칸도 읽으세요. 역할 프롬프트 행은 status가 없어 코드가 complete로 가정하고 넣은 결과입니다. 작업 계약 행에 `등록되지 않은 근거`가 나왔다면 계약을 받은 모델도 사건 번호처럼 조회 참조가 아닌 값을 근거 칸에 넣을 수 있다는 뜻입니다. 계약은 어길 수 없는 규칙이 아니라 지시입니다. 그래서 다음 절에서 코드로 검사합니다.

역할 프롬프트가 무의미하다는 뜻이 아닙니다. 역할은 말투와 관점을 정하는 데 쓸모가 있습니다. 다만 역할 이름만으로 업무가 완성되지는 않습니다. 그리고 이것은 1회 관찰입니다. 같은 지시를 다시 돌리면 답이 달라질 수 있어 이 한 번으로 어느 쪽이 얼마나 낫다고 말할 수 없습니다. 차이가 어떤 칸에서 생기는지를 읽는 데에만 쓰세요.

## 3. Context: 전달 계약과 손실

**문제.** Worker 사이에 무엇을 넘기느냐가 다음 단계가 볼 수 있는 세계를 정합니다. 2절의 계약이 Worker 한 명에게 일을 시키는 약속이라면, 이번 절의 계약은 Worker와 Worker, Worker와 작성자 사이의 약속입니다. 입력 계약 `WorkerTask`는 일을 나눠 보내는 쪽(이 과정에서는 그래프의 분배 노드)이 Worker에게 주는 것이고, 반환 계약 `WorkerResult`는 Worker가 돌려주는 것입니다.

입력에 `allowed_tools`가 있는 이유를 스키마를 보며 생각해 보세요. 작업마다 필요한 권한을 작업 자체에 적어 두면 무엇을 볼 수 있고 무엇을 할 수 있는지를 실행 전에 검토할 수 있습니다. 결과를 받은 쪽도 이 Worker가 원래 볼 수 없는 사실을 어떻게 알았는지 물을 수 있습니다. 반환 계약이 요약 대신 `facts`, `evidence_refs`, `uncertainty`를 따로 두는 이유는 다음 단계가 사실과 근거를 대조하고 모르는 것을 물려받게 하기 위해서입니다.

In [ ]:
def schema_rows(contract) -> list[dict]:
    schema = contract.model_json_schema()
    return [{"계약": contract.__name__, "필드": name, "타입": spec.get("type", "object"), "허용 값": spec.get("enum", ""),
             "필수": name in schema.get("required", []), "기본값": spec.get("default", "")}
            for name, spec in schema["properties"].items()]

show_table(schema_rows(WorkerTask) + schema_rows(WorkerResult))
print("allowed_tools를 작업에 두면 이 작업의 권한을 실행 전에 검토할 수 있습니다.")
print("모델 호출 0회")

In [ ]:
# 실패 재현: 잘못된 Worker 결과 네 개를 스키마와 근거 검증에 차례로 넣습니다.
check_desk = make_desk(case, label="check")
check_desk.tools("outage")[0].invoke({"month": "2026-08"})  # 실제 조회로 F-AV-2026-08을 근거로 등록합니다.
refs = check_desk.known_refs()
bad_results = {"없는 status": {"status": "done", "facts": ["가용률 99.3"], "evidence_refs": ["F-AV-2026-08"]},
               "없는 근거 ID": {"status": "complete", "facts": ["가용률 99.3"], "evidence_refs": ["lookup_outage: 8월 장애"]},
               "근거 없는 complete": {"status": "complete", "facts": ["가용률 99.3"], "evidence_refs": []},
               "실제 근거에 틀린 사실": {"status": "complete", "facts": ["가용률 99.95"], "evidence_refs": ["F-AV-2026-08"]}}
rows = []
for name, data in bad_results.items():
    try:
        result, schema = WorkerResult(**data), "통과"
    except ValidationError as error:
        result, schema = None, f"거절: {error.errors()[0]['type']}"
    problems = validate_result(result, refs) if result else None
    rows.append({"잘못된 결과": name, "Pydantic 스키마": schema, "validate_result": "검사 전에 멈춤" if result is None else problems or "통과",
                 "거절한 층": "스키마" if result is None else "근거 검증" if problems else "어느 층도 거절하지 않음"})
show_table(rows)
print("등록된 근거:", sorted(refs), "| 모델 호출 0회")

표의 위 세 행은 각각 다른 층이 거절합니다. 스키마는 모양을 보고, `validate_result`는 이번 실행에 등록된 근거인지를 봅니다. 두 번째 행의 `"lookup_outage: 8월 장애"`는 지어낸 예가 아닙니다. 이 과정을 준비하며 돌린 시험 실행에서 실제 모델이 `evidence_refs`에 이런 값을 냈습니다. 문자열 칸이므로 스키마는 통과합니다.

네 번째 행을 주의해서 읽으세요. 근거 ID는 진짜이고 사실은 틀렸습니다. 검증을 두 겹 두어도 이 행은 지나갑니다. 근거 검증은 인용이 이번 실행에 있었는지를 볼 뿐, 문장이 그 근거와 뜻이 같은지를 판단하지 않습니다. 그래서 인용이 있는 결과라도 사람이 원문과 대조하는 일이 남습니다.

이제 전달 형태로 넘어갑니다. 투자 심사 Agent를 설계하던 판서에 이런 메모가 있습니다. (원본은 저장소의 `Day03/Day03_01.png`입니다. 용량이 커서 이 폴더에 복사하지 않고 말로 옮겼습니다.) "개별 에이전트가 핵심 1줄만 → 80점", "Context가 넘쳐나는데 어떻게 요약할 것인가?" 요약은 입력을 줄여 주지만 무엇을 지우는지 말해 주지 않습니다. 이 메모를 T-1001에서 직접 재 봅니다.

**먼저 예측하세요.** T-1001의 크레딧은 기본 이용료를 기준으로 계산하고(`sla-a-v2.md:11`), 신청 기한은 추가 약정이 정합니다(`sla-a-amendment.md:6`). Worker마다 핵심 1줄만 작성자에게 넘기면 무엇이 먼저 사라질까요? 1,200,000원을 계산 기준으로 잘못 쓸까요, 아니면 금액을 모른다고 할까요? 신청 기한은 며칠로 답할까요? 답을 적은 뒤 실행하세요.

In [ ]:
announce("T-1001 Worker 3개 실행", model_calls=12)
case_desk, case_meter = make_desk(case, label="payloads"), ModelMeter()

def run_role(role: str) -> tuple[str, WorkerResult]:
    task = WorkerTask(task_id=role, goal=f"{ROLE_GOALS[role]}\n고객 요청: {case['question']}", allowed_tools=sorted(ROLE_TOOLS[role]))
    return role, run_worker(task, case_desk, model, config={"callbacks": [case_meter]})

with ThreadPoolExecutor(max_workers=3) as pool:
    worker_results = dict(pool.map(run_role, ROLES))
# 뒤 셀들은 이 결과를 값으로 읽습니다. 예산의 시간 한도가 뒤 셀까지 이어지지 않게 Desk를 다시 쓰지 않습니다.
case_refs, case_evidence = case_desk.known_refs(), dict(case_desk.context.evidence)
case_quotes = {**dict(case_desk.fact_refs), **{ref: item.quote for ref, item in case_evidence.items()}}
results_hash = plan_hash({role: result.model_dump() for role, result in worker_results.items()})
show_table([{"작업": role, "status": r.status, "사실 수": len(r.facts), "근거 수": len(r.evidence_refs), "불확실성 수": len(r.uncertainty),
             "검증 문제": validate_result(r, case_refs) or "없음", "오류": r.error} for role, r in worker_results.items()])
print(timeline(case_desk.trace.spans))
print("모델 호출:", case_meter.totals()["model_calls"], "| Worker 결과 지문:", results_hash)

In [ ]:
def quotes_of(result: WorkerResult) -> dict[str, str]:
    return {ref: case_quotes[ref] for ref in result.evidence_refs if ref in case_quotes}

def summary_line(result: WorkerResult) -> str:
    # 코드가 만듭니다. 요약에 모델을 쓰면 비교할 변수가 하나 더 늘어납니다.
    return result.facts[0] if result.facts else f"(사실 없음: {result.status})"

evidence_shape = {role: r.model_dump(include={"status", "facts", "evidence_refs", "uncertainty", "error"}) for role, r in worker_results.items()}
payloads = {"1줄 요약": {role: summary_line(r) for role, r in worker_results.items()},
            "사실+근거+불확실성": evidence_shape,
            "원문 포함": {role: {**evidence_shape[role], "quotes": quotes_of(r)} for role, r in worker_results.items()}}
payload_text = {name: json.dumps(p, ensure_ascii=False, indent=1) for name, p in payloads.items()}
show_table([{"전달물": name, "바이트": len(text.encode("utf-8")), "인용 근거 참조": len(set(REF.findall(text))),
             "불확실성 칸": "uncertainty" in text, "Worker 결과 지문": results_hash} for name, text in payload_text.items()])
print(payload_text["1줄 요약"])
print("모델 호출 0회. Worker를 다시 돌리지 않습니다. 전달 형태 하나만 바꾸기 위해서입니다.")

In [ ]:
announce("같은 작성자에게 전달물 3종 주기", model_calls=3)
WRITER_PROMPT = ("고객에게 보낼 답을 씁니다. {scope}\n{guide}\n"
                 "환불이 맞다면 proposal에 action, payment_id, amount, reason을 쓰세요. 실행 여부는 코드 게이트가 정합니다.\n"
                 "아래 전달물이 당신이 가진 자료의 전부입니다. 보상 금액은 전달물로 계산할 수 있으면 계산하고, 확인되지 않는 것은 uncertainty에 쓰세요.\n"
                 "evidence_refs에는 전달물에 적힌 fact_ref와 evidence_id만 넣으세요.\n고객 요청: {question}\n전달물:\n{payload}")
guide = "\n".join(OUTCOME_GUIDE.splitlines()[:3])
writer, writer_meter, writer_rows = model.with_structured_output(FinalAnswer, method="json_schema"), ModelMeter(), []
for name, text in payload_text.items():
    prompt = WRITER_PROMPT.format(scope=scope_line(case_desk), guide=guide, question=case["question"], payload=text)
    try:
        final, error = writer.invoke(prompt, config={"callbacks": [writer_meter]}), ""
    except Exception as failure:  # 실패한 호출도 표에 행으로 남깁니다.
        final, error = None, type(failure).__name__
    writer_rows.append({"전달물": name, "final": final, "오류": error, "입력 토큰": writer_meter.calls[-1]["input_tokens"], "지문": results_hash})
    print(f"--- {name}\n{final.text if final else error}\n불확실성: {final.uncertainty if final else ''}")
print("모델 호출:", writer_meter.totals()["model_calls"], "| usage 누락:", writer_meter.totals()["usage_missing"])

In [ ]:
def writer_row(item: dict) -> dict:
    final = item["final"]
    refs = final.evidence_refs if final else []
    return {"structure": item["전달물"], "case_id": case["id"], "answer": final.text if final else "", "uncertainty": final.uncertainty if final else [],
            "outcome": final.outcome if final else None, "cited_docs": sorted({case_evidence[r].doc_id for r in refs if r in case_evidence}),
            "ledger_refunds": 0, "status": "ok" if final else "model_error"}

CONDITION = re.compile(r"계산 기준|포함하지 않|(?:사용량|사용료).{0,30}제외")
DEADLINE_20, DEADLINE_15 = re.compile(r"(?<!\d)20일|09-20"), re.compile(r"(?<!\d)15일|09-15")
checked, condition_rows = [], []
for item in writer_rows:
    row, final = writer_row(item), item["final"]
    check = evaluate_row(row, gold)
    refs = final.evidence_refs if final else []
    answer, given = row["answer"] + " ".join(row["uncertainty"]), payload_text[item["전달물"]]
    checked.append({"전달물": item["전달물"], "입력 토큰": item["입력 토큰"], "자동 검사": check["자동 검사"], "outcome": check["실제 outcome"], "누락 사실": check["누락 사실"],
                    "금지 포함": check["금지 포함"], "누락 문서": check["누락 문서"], "유효 인용": sum(r in case_refs for r in refs), "없는 인용": [r for r in refs if r not in case_refs]})
    condition_rows.append({"전달물": item["전달물"], "전달물에 계산 기준 조건": bool(CONDITION.search(given)), "답에 계산 기준 조건": bool(CONDITION.search(answer)), "답에 150,000": "150,000" in answer or "150000" in answer,
                           "전달물에 20일 기한": bool(DEADLINE_20.search(given)), "답에 20일 기한": bool(DEADLINE_20.search(answer)), "답에 15일 기한": bool(DEADLINE_15.search(answer))})
show_table(checked)
show_table(condition_rows)
print("자동 검사는 문자열 포함 검사입니다. 표현이 다르면 놓칠 수 있으니 위에 출력된 답과 함께 읽으세요. 모델 호출 0회")

In [ ]:
policy_result = worker_results["policy"]
policy_quotes = quotes_of(policy_result)
full_size = trim_payload(policy_result, policy_quotes, 10**9)["bytes"]
rows = []
for limit in (full_size, int(full_size * 0.6), int(full_size * 0.3), 120):
    trimmed = trim_payload(policy_result, policy_quotes, limit)
    kept = trimmed["payload"]
    rows.append({"예산 바이트": limit, "남은 바이트": trimmed["bytes"], "예산 초과": trimmed["over_budget"], "뺀 항목 수": len(trimmed["dropped"]),
                 "뺀 항목": trimmed["dropped"][:3], "남은 사실": len(kept["facts"]), "남은 원문": len(kept["quotes"]),
                 "남은 근거 참조": len(kept["evidence_refs"]), "불확실성 유지": kept["uncertainty"] == policy_result.uncertainty})
show_table(rows)
print("줄이는 순서는 원문, 그다음 사실입니다. 근거 참조와 불확실성은 끝까지 남깁니다. 모델 호출 0회")

In [ ]:
# 실패 재현: 바이트에서 그냥 자르면 예외 문장이 사라집니다.
cut_desk = make_desk(case, label="cut")
clause = cut_desk.context.read_more("sla-a-v2", 12, 3)["evidence"]["quote"]  # sla-a-v2.md 12~14행의 제외 시간 조항입니다.
EXCEPTION = "단, 월 누적 120분을 초과한 점검 시간은 포함한다"
raw = clause.encode("utf-8")
rows = []
for limit in (len(raw), 180, 120, 100, 60):
    kept = raw[:limit].decode("utf-8", errors="ignore")
    rows.append({"예산 바이트": limit, "예외 문장 남음": EXCEPTION in kept, "남은 끝부분": kept[-34:].replace("\n", " ")})
show_table(rows)

# 검색에서 빠진 문서는 뒤에서 되살릴 수 없습니다. 줄이는 쪽은 받은 것만 줄일 수 있습니다.
query = "서비스 크레딧 신청 기한"
shown = [{"검색 limit": limit, "찾은 문서": sorted({e["doc_id"] for e in search["evidence"]}), "찾지 못한 관련 문서": search["related_docs"]}
         for limit in (1, 6) for search in [make_desk(case, label="search").context.search(query, limit=limit)]]
show_table(shown)
print("모델 호출 0회")

**관찰할 것.** 두 가지 수를 구분해서 읽으세요. `입력 토큰`은 작성자 호출 전체의 입력이고, 전달물의 `바이트`는 그중 근거 Context의 크기입니다. 지시문과 스키마는 세 호출이 같으므로 토큰 차이는 전달물에서 옵니다. 토큰이 줄었는데 `누락 사실`이나 `누락 문서`가 늘었다면 두 변화를 항상 함께 보고하세요. 토큰이 줄었다는 절반만 보고하면 손해를 이득으로 잘못 적게 됩니다.

`자동 검사` 표는 1줄 요약 행부터 읽습니다. `금지 포함`에 `180000`이 있으면 사용량 초과 제외 조건이 사라진 것이고, `누락 사실`에 `150000`이 있고 금지 표현이 없다면 작성자가 금액을 정직하게 비워 둔 것입니다. 답이 그럴듯해도 `유효 인용`이 0이면 다음 단계가 대조할 근거가 없습니다. `없는 인용` 칸에 값이 있다면 값을 직접 읽으세요. 1줄 요약에는 참조가 하나도 없는데도 전달물의 키 이름(`billing`)을 근거처럼 적을 수 있습니다. 다른 행에서는 Worker가 근거 칸에 넣은 사건 번호(`INC-311`)가 작성자의 인용까지 그대로 흘러갈 수 있습니다. Worker 표의 `검증 문제` 칸에 같은 값이 있는지 대조해 보세요. 검증 결과를 보고만 넘기면 잘못된 참조가 다음 단계로 전달됩니다. 어디서 걸러야 할지는 6절의 합류에서 다시 생각합니다.

두 번째 표에서 `전달물에`로 시작하는 칸이 거짓이면 전달 전에 이미 조건을 잃은 것이고, 참인데 `답에` 칸이 거짓이면 전달은 됐지만 작성자가 쓰지 않은 것입니다. 원인이 다르므로 고칠 곳도 다릅니다. 이 표도 문자열 검사이므로 문장의 뜻은 위에 출력된 답을 직접 읽어 확인하세요.

두 표가 어긋나는 행도 찾아보세요. `누락 사실`에 `9월20일`이 있는데 `답에 20일 기한`이 참이라면, 답이 `2026-09-20`처럼 다른 표기로 쓴 것입니다. 답은 맞는데 자동 검사가 놓친 경우입니다. 그래서 칸 이름이 `정답`이 아니라 `자동 검사`입니다. 이 칸의 거짓을 바로 오답으로 옮겨 적지 마세요.

사실+근거+불확실성 행과 원문 포함 행의 `자동 검사`가 다르면 먼저 그 차이가 표기 때문인지 확인하세요. 표기 차이를 빼고 결과가 같다면, 원문을 더 넘긴 입력 토큰이 이번 1회에서는 필수 사실을 늘리지 못한 것입니다. 그렇다고 원문이 항상 낭비는 아닙니다. Worker가 사실을 잘못 옮겼을 때 작성자가 대조할 수 있는 유일한 자료이기 때문입니다. 세 행의 결과가 모두 같게 나왔다면 Worker가 첫 번째 사실에 핵심을 많이 몰아 썼을 수 있습니다. 이 경우 "요약이 괜찮다"가 아니라 "이번 1회에서는 잃지 않았다"로 기록하세요.

이 작성자에게는 계산 도구가 없습니다. 판서의 다른 질문이 여기로 이어집니다. "숫자가 들어가면 LLM은 거짓말을 할 가능성이 몇 %?", "Tool: 계산기". 곱셈은 코드에게 맡기고 비율을 고르는 판단만 모델이 하는 것이 8절의 `calculate_credit`입니다.

`trim_payload` 표에서는 `남은 원문`이 먼저 줄고 `남은 사실`이 뒤에 줄어듭니다. 근거 참조와 불확실성은 끝까지 유지합니다. 이 순서는 `policy` Worker가 근거 참조를 돌려줬을 때만 보입니다. Worker 표에서 `policy`의 `근거 수`가 0이면 줄일 원문과 근거 참조가 처음부터 없습니다. `complete`였다면 `검증 문제`에 `근거 없이 complete입니다.`도 함께 나옵니다. 그 실행에서는 `남은 원문`과 `남은 근거 참조`가 모든 행에서 0이고 `남은 사실`만 줄어드니, 원문이 먼저 빠지는 순서를 이 표로 확인했다고 적지 마세요. 가장 작은 예산에서 `예산 초과`가 참이면 근거 참조와 불확실성이 이미 예산보다 크다는 뜻입니다. 빠뜨리지 않고 넘겼다고 알려 주는 것이지 예산을 지킨 것은 아닙니다. 항목 단위로 줄이면 `뺀 항목` 칸에 무엇을 뺐는지 남습니다.

바이트 자르기 표는 다릅니다. `sla-a-v2.md:13`의 "단, 월 누적 120분을 초과한 점검 시간은 포함한다"가 사라져도 남은 문장은 멀쩡해 보입니다. 뜻이 바뀌었는데 오류가 나지 않습니다. 줄이는 쪽은 받은 것 안에서만 일할 수 있습니다. 검색 표의 `limit=1` 행처럼 추가 약정이 처음부터 들어오지 않았다면 어떤 요약도 그 약정을 복원할 수 없습니다. 이 손실은 검색 설계로 돌아가 고쳐야 합니다.

## 4. Harness: 공유 예산

**문제.** Worker 셋이 각자 호출 한도를 가지면 전체 비용은 누가 막을까요? 재시도와 병렬이 더해지면 호출 수는 쉽게 늘어납니다. 모델은 자기가 얼마를 썼는지 모르고, 알더라도 멈춰 달라는 지시는 강제력이 없습니다. 실행, 검증, 중단, 복구는 모델 주변 시스템의 책임입니다.

`Budget`은 한 실행의 호출을 하나의 카운터로 셉니다. 먼저 모델 없이 동작만 확인합니다. 스레드 3개가 각 3번씩 소모를 시도하고 한도는 5입니다. 표의 `누적 model`이 한도를 넘는 행이 있는지 찾아보세요. 잠금이 있어 여러 스레드가 동시에 세어도 한도를 넘지 않아야 합니다.

In [ ]:
shared = Budget(max_model_calls=5)
log, log_lock = [], threading.Lock()

def consumer(name: str) -> None:
    for attempt in range(1, 4):
        time.sleep(0.01 * attempt)
        try:
            shared.consume("model", worker=name)
            outcome = "허용"
        except BudgetExceeded as error:
            outcome = f"BudgetExceeded: {error}"
        with log_lock:
            log.append({"순서": len(log) + 1, "스레드": name, "시도": attempt, "결과": outcome, "누적 model": shared.counts.get("model", 0)})

threads = [threading.Thread(target=consumer, args=(name,)) for name in ("billing", "policy", "outage")]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
show_table(log)
print("허용:", sum(r["결과"] == "허용" for r in log), "| 거절:", sum(r["결과"] != "허용" for r in log), "| 모델 호출 0회")

In [ ]:
# 실패 재현: 모델 호출 예산 2회로 Worker 셋을 차례로 실행합니다.
announce("예산 2회 Worker 실행", model_calls=2)
small_desk, small_meter = make_desk(case, budget={"max_model_calls": 2}, label="small"), ModelMeter()
rows, stopped = [], False
for role in ROLES:
    task = WorkerTask(task_id=role, goal=f"{ROLE_GOALS[role]}\n고객 요청: {case['question']}")
    if stopped:
        rows.append({"작업": role, "상태": "시작하지 않음", "오류": "앞 작업에서 예산 소진", "누적 model": small_desk.budget.counts.get("model", 0)})
        continue
    try:
        status, error = run_worker(task, small_desk, model, config={"callbacks": [small_meter]}).status, ""
    except BudgetExceeded as failure:
        status, error, stopped = "budget_exceeded", str(failure), True
    rows.append({"작업": role, "상태": status, "오류": error, "누적 model": small_desk.budget.counts.get("model", 0)})
show_table(rows)
report = small_desk.budget.report()
print("budget.report() counts:", report["counts"], "| 경과 초:", report["elapsed_seconds"], "| 실제 모델 호출:", small_meter.totals()["model_calls"])
show_table([{k: e.get(k, "") for k in ("event", "seconds", "name")} for e in report["events"] if e["event"] != "usage"])

**관찰할 것.** 첫 번째 표에서 한도를 넘는 `누적 model`이 없는지 확인하세요. 어떤 스레드가 거절당하는지는 실행마다 달라질 수 있지만, 허용 건수는 한도와 같아야 합니다. 두 번째 표는 `상태` 칸을 읽습니다. 예산을 다 쓴 작업이 어느 것인지, 그 앞 작업의 결과는 살아 있는지 보세요. `budget_exceeded`는 조용히 넘어갈 일이 아니라 드러내야 할 실패입니다. 셋째 표는 예산을 어떤 호출이 소진했는지 시간 순서로 보여 줍니다.

여기서 할 수 있는 가장 나쁜 대응은 숫자를 올려 실패를 감추는 것입니다. 그 전에 물어야 합니다. 이 호출들은 다 필요했는가? 같은 도구를 반복해 부르지는 않았는가? 한 Worker가 다른 Worker의 예산까지 먹었는가? 근거를 갖고 한도를 정했다면 그때 한도를 바꾸세요. 그 이유도 함께 기록합니다.

## 5. Graph와 Loop: 실패했다고 무조건 다시 부르지 않습니다

**문제.** 가장 흔한 복구 코드는 `try`로 잡고 다시 호출하는 것입니다. 시간 초과라면 맞을 수 있습니다. 그런데 근거가 부족한 답을 그대로 다시 호출하면 비슷한 답에 비용만 더 듭니다. 정책상 환불이 불가능한 요청을 다시 호출하면 같은 거절이 나오거나, 더 나쁘게는 다른 경로로 뚫으려는 시도가 됩니다. 예산을 넘었는데 다시 부르면 예산이 뜻을 잃습니다.

실패의 **종류**가 다음 행동을 정합니다. 이 분류는 모델이 아니라 코드가 합니다.

**먼저 예측하세요.** 다음 표의 `다음 행동`을 가리고 근거 부족, 정책 불가, 시스템 오류, 예산 초과 각각의 다음 행동을 먼저 적으세요. 재시도 기회가 남았을 때와 다 썼을 때를 나눠 적습니다.

In [ ]:
print(inspect.getsource(classify_failure))
print(inspect.getsource(next_action))
show_table([{"실패 종류": kind.value, "재시도 남음": retries, "예산 남음": left, "다음 행동": next_action(kind, retries_left=retries, budget_left=left)}
            for kind in FailureKind for retries, left in ((1, True), (0, True), (1, False))])
print("모델 호출 0회")

In [ ]:
class FailState(TypedDict, total=False):
    step: int
    retries_left: int
    kind: str
    action: str
    path: list[str]

def build_failure_graph(act, decide=next_action, retries: int = 1):
    # act(step)가 (오류, 최종 답, 문제 목록)을 돌려줍니다. 분류와 다음 행동은 코드가 정합니다.
    def attempt(state: FailState) -> dict:
        error, final, problems = act(state.get("step", 0))
        kind = classify_failure(error, final, problems)
        action = decide(kind, retries_left=state["retries_left"], budget_left=not isinstance(error, BudgetExceeded))
        return {"step": state.get("step", 0) + 1, "kind": kind.value, "action": action,
                "retries_left": state["retries_left"] - (action in {"retry", "reinvestigate"}),
                "path": [*state.get("path", []), f"{kind.value} → {action}"]}
    graph = StateGraph(FailState)
    graph.add_node("attempt", attempt)
    graph.add_edge(START, "attempt")
    graph.add_conditional_edges("attempt", lambda s: "attempt" if s["action"] in {"retry", "reinvestigate"} else END, ["attempt", END])
    return graph.compile(), {"retries_left": retries}

print("상태 칸:", list(FailState.__annotations__), "| 모델 호출 0회")

In [ ]:
INJECT = {"pass": (None, FinalAnswer(outcome="answered", text="(주입)", evidence_refs=["F-1"]), []),
          "insufficient": (None, FinalAnswer(outcome="insufficient", text="(주입)"), []),
          "denied": (None, FinalAnswer(outcome="denied", text="(주입)"), []),
          "timeout": (TimeoutError("주입한 시간 초과"), None, []),
          "budget": (BudgetExceeded("주입한 예산 초과"), None, [])}
SCENARIOS = {"근거 부족": ["insufficient", "pass"], "정책 불가": ["denied", "pass"], "시스템 오류": ["timeout", "pass"], "예산 초과": ["budget", "pass"]}

def run_scenarios(decide=next_action, retries: int = 1) -> list[dict]:
    rows = []
    for name, script in SCENARIOS.items():
        app, start = build_failure_graph(lambda step, s=script: INJECT[s[min(step, len(s) - 1)]], decide, retries)
        end = app.invoke(start)
        rows.append({"주입": name, "시도 수": end["step"], "경로": " | ".join(end["path"]), "끝난 행동": end["action"]})
    return rows

show_table(run_scenarios())
print("모델 호출 0회")

In [ ]:
calls = Counter()

def run_policy_graph(node, retry_on=TimeoutError, max_attempts: int = 3) -> str:
    graph = StateGraph(FailState)
    graph.add_node("work", node, retry_policy=RetryPolicy(retry_on=retry_on, max_attempts=max_attempts, initial_interval=0.01, jitter=False))
    graph.add_edge(START, "work")
    graph.add_edge("work", END)
    try:
        return f"끝남: {graph.compile().invoke({'step': 0}).get('kind')}"
    except Exception as error:
        return f"{type(error).__name__}: {str(error)[:50]}"

def recovers(state):
    calls["시간 초과 두 번 뒤 회복"] += 1
    if calls["시간 초과 두 번 뒤 회복"] < 3:
        raise TimeoutError("주입한 시간 초과")
    return {"kind": "pass"}

def always_times_out(state):
    calls["계속 시간 초과"] += 1
    raise TimeoutError("주입한 시간 초과")

def insufficient_answer(state):
    calls["근거 부족을 정상 값으로 반환"] += 1
    return {"kind": "insufficient"}

outcomes = {"시간 초과 두 번 뒤 회복": run_policy_graph(recovers), "계속 시간 초과": run_policy_graph(always_times_out),
            "근거 부족을 정상 값으로 반환": run_policy_graph(insufficient_answer)}
show_table([{"실험": name, "노드 실행 횟수": calls[name], "결과": outcomes[name]} for name in outcomes])
print("모델 호출 0회")

In [ ]:
send_graph = StateGraph(FailState)
send_graph.add_node("work", lambda state: {"kind": "pass"})
send_graph.add_conditional_edges(START, lambda state: [Send("work", {"step": 0}, timeout=0.2)], ["work"])
send_graph.add_edge("work", END)
try:
    print(send_graph.compile().invoke({"step": 0}))
except ValueError as error:
    print("Send(timeout=)을 동기 노드에 쓰면:", type(error).__name__, "-", error)
print("모델 호출 0회")

`RetryPolicy`와 `next_action`은 경쟁 관계가 아니라 맡는 범위가 다릅니다. `RetryPolicy`는 **같은 노드**를 **예외가 났을 때** 다시 실행합니다. 위 표에서 근거 부족을 정상 값으로 돌려준 노드는 한 번만 돕니다. 예외가 아니기 때문입니다. 재조사, 미실행 종료, 사람에게 넘기기는 다른 노드로 가는 일이므로 그래프의 조건부 간선이 맡아야 합니다.

`Send(timeout=)`은 동기 노드에서 `ValueError`가 납니다. 동기 Python 코드는 프로세스 안에서 안전하게 취소할 수 없기 때문입니다. 이 과정의 노드는 동기이므로 시간 제한은 `Budget`이 맡습니다. `Budget`의 시간 한도는 다음 소모 시점에 검사하는 방식이라 이미 돌고 있는 호출을 끊지는 않습니다. 개별 호출의 제한 시간은 `make_model`의 `timeout`이 맡습니다.

이제 실제 Worker로 같은 복구를 합니다. 합류 노드가 `join_results`로 상태를 읽고, `next_action`이 `retry`라고 할 때만 실패한 작업을 새 키 `outage#2`로 다시 보냅니다. `lookup_availability`에는 한 번만 시간 초과를 주입합니다. 이 그래프는 작성자 없이 Worker까지만 돕니다. 복구 경로에만 집중하기 위해서입니다.

In [ ]:
announce("시간 초과 1회 주입 뒤 Worker 복구", model_calls=12)
fault_desk, fault_meter = make_desk(case, faults={"lookup_availability": "timeout_once"}, label="fault"), ModelMeter()

class RecoverState(TypedDict, total=False):
    results: Annotated[dict, merge_results]
    task: dict

def recover_worker(state: RecoverState) -> dict:
    task = WorkerTask.model_validate(state["task"])
    return {"results": {task.task_id: run_worker(task, fault_desk, model, config={"callbacks": [fault_meter]}).model_dump()}}

def after_recover_join(state: RecoverState):
    results = {k: WorkerResult.model_validate(v) for k, v in state["results"].items()}
    info = join_results(results, list(ROLES))
    retries_left = 0 if any("#" in k for k in results) else 1
    action = next_action(FailureKind.SYSTEM_ERROR, retries_left=retries_left, budget_left=True) if info["failed"] else "respond"
    fault_desk.trace.event("join", "code", failed=info["failed"], action=action)
    if action != "retry":
        return END
    return [Send("worker", {"task": WorkerTask(task_id=f"{t}#2", attempt=2, goal=f"{ROLE_GOALS[t]}\n고객 요청: {case['question']}").model_dump()}) for t in info["failed"]]

recover_graph = StateGraph(RecoverState)
recover_graph.add_node("worker", recover_worker)
recover_graph.add_node("join", lambda state: {"results": {}})
recover_graph.add_conditional_edges(START, lambda state: [Send("worker", {"task": WorkerTask(task_id=r, goal=f"{ROLE_GOALS[r]}\n고객 요청: {case['question']}").model_dump()}) for r in ROLES], ["worker"])
recover_graph.add_edge("worker", "join")
recover_graph.add_conditional_edges("join", after_recover_join, ["worker", END])
recovered = recover_graph.compile().invoke({"results": {}}, config={"recursion_limit": 20})
print("모델 호출:", fault_meter.totals()["model_calls"], "| 결과 키:", sorted(recovered["results"]))

In [ ]:
fields = ("at", "kind", "actor", "fault", "failed", "action", "month", "invoice_id", "query")
show_table([{k: e.get(k, "") for k in fields} for e in fault_desk.trace.events])
tool_counts = Counter(e["name"] for e in fault_desk.budget.report()["events"] if e["event"] == "tool")
worker_runs = Counter(s["actor"] for s in fault_desk.trace.spans)
show_table([{"구분": "도구", "이름": name, "횟수": count} for name, count in sorted(tool_counts.items())] +
           [{"구분": "Worker 구간", "이름": name, "횟수": count} for name, count in sorted(worker_runs.items())])
print(timeline(fault_desk.trace.spans))
show_table(join_results({k: WorkerResult.model_validate(v) for k, v in recovered["results"].items()}, list(ROLES))["rows"])

In [ ]:
always_desk = make_desk(case, faults={"lookup_availability": "timeout_always"}, label="always")
availability = always_desk.tools("outage")[0]

def call_availability(step: int) -> tuple:
    # 실제 도구를 부릅니다. 모델은 없습니다. 제어 경로만 확인합니다.
    try:
        availability.invoke({"month": "2026-08"})
    except TimeoutError as error:
        return error, None, []
    return None, FinalAnswer(outcome="answered", text="(조회됨)", evidence_refs=["F-AV-2026-08"]), []

always_app, always_start = build_failure_graph(call_availability)
always_end = always_app.invoke(always_start)
show_table([{"주입": "timeout_always", "시도 수": always_end["step"], "경로": " | ".join(always_end["path"]), "끝난 행동": always_end["action"]}])
print("주입 기록:", Counter(e["kind"] for e in always_desk.trace.events), "| 모델 호출 0회")

**관찰할 것.** 복구의 비용과 보존의 효과를 함께 읽으세요. 사건 표에서 `fault_injected` 다음에 `join`의 `action`이 무엇인지 찾습니다. `Worker 구간` 표에 `outage`와 `outage#2`가 따로 있으면 재시도가 돈 것입니다. `billing`과 `policy`가 각각 1회면 완료된 Worker는 다시 돌지 않았습니다. 재시도의 비용은 `outage#2` 구간과 그만큼의 모델 호출이고, 그 대신 다른 Worker의 호출을 다시 쓰지 않습니다. `outage#2`가 없다면 모델이 도구를 부르지 않았거나 다른 경로로 끝난 것입니다. `lookup_availability` 호출 횟수와 맨 아래 상태 표로 어느 쪽인지 확인하세요.

두 표의 수가 다를 수 있습니다. `도구` 행은 시작한 호출을 세므로 주입으로 실패한 호출까지 포함하고, 사건 표의 `tool` 행은 주입을 지나간 호출만 남습니다. 모델이 한 메시지에서 도구를 둘 불렀다면 하나만 실패해도 그 Worker는 `failed`가 됩니다. 함께 성공한 조회 결과도 Worker 결과로는 돌아오지 않습니다. `month` 칸에서 모델이 어느 달을 물었는지도 보세요. 이 값은 모델이 정합니다. 맨 아래 표에서 `outage`의 상태가 `uncertain`일 수 있습니다. 복구는 실행이 끝났다는 뜻이지 작업이 답을 다 찾았다는 뜻이 아닙니다.

`timeout_always` 행은 `escalate`로 끝나야 합니다. 재시도 기회를 다 쓰면 다시 호출하지 않고 사람에게 넘깁니다. 이것은 제어 경로의 확인이지 장애율 측정이 아닙니다. 주입한 오류는 실제 장애가 얼마나 자주 나는지 아무것도 말해 주지 않습니다.

## 6. 부분 실패와 합류

**문제.** 병렬 Worker 셋 중 하나만 실패했습니다. 그래프는 예외 없이 다음 노드로 갑니다. 합류 노드는 무엇을 해야 할까요? 03에서 만든 병렬은 모두 성공한다고 가정했습니다. 운영에서는 일부만 끝나는 경우가 더 어렵습니다. 전체가 실패하면 누구나 알아차리지만 일부만 실패하면 정상처럼 보일 수 있기 때문입니다.

**먼저 예측하세요.** `outage` Worker가 실패했는데 합류가 성공한 결과만 모아 작성 노드로 넘깁니다. 작성 노드의 입력에 실패를 알 수 있는 흔적이 남을까요? 그 입력으로 쓴 답은 고객에게 어떻게 보일까요? 가용률이 없는 이유를 작성자가 알 수 있을까요?

이 절의 Worker는 일반 함수이고 정해 둔 결과를 돌려줍니다. 합류의 제어를 보는 것이 목적이므로 모델이 필요 없습니다.

In [ ]:
# 실패 재현: 성공한 결과만 모아 넘기는 합류는 실패를 지웁니다.
SAMPLE = {"billing": {"status": "complete", "facts": ["P-9002 1,200,000원은 P-9001과 같은 청구서의 나중 결제입니다."], "evidence_refs": ["F-P-9002"]},
          "policy": {"status": "complete", "facts": ["99.0% 이상 99.5% 미만 구간의 크레딧은 15%입니다."], "evidence_refs": ["E-sample"]},
          "outage": {"status": "failed", "error": "TimeoutError"},
          "outage#2": {"status": "complete", "facts": ["2026-08 가용률은 99.3입니다."], "evidence_refs": ["F-AV-2026-08"]}}
worker_counts = Counter()

class JoinState(TypedDict, total=False):
    results: Annotated[dict, merge_results]
    task: str
    writer_input: dict

def sample_worker(state: JoinState) -> dict:
    task_id = state["task"]
    worker_counts[task_id] += 1
    return {"results": {task_id: WorkerResult(task_id=task_id, **SAMPLE[task_id]).model_dump()}}

def silent_join(state: JoinState) -> dict:
    return {"writer_input": {k: v["facts"] for k, v in state["results"].items() if v["status"] == "complete"}}

silent = StateGraph(JoinState)
silent.add_node("worker", sample_worker)
silent.add_node("join", silent_join)
silent.add_conditional_edges(START, lambda state: [Send("worker", {"task": r}) for r in ROLES], ["worker"])
silent.add_edge("worker", "join")
silent.add_edge("join", END)
silent_end = silent.compile().invoke({"results": {}})
print("작성 노드의 입력:", json.dumps(silent_end["writer_input"], ensure_ascii=False))
print("입력에 outage 언급:", "outage" in json.dumps(silent_end["writer_input"]), "| 예외 발생: 없음 | 모델 호출 0회")

In [ ]:
partial = {"billing": WorkerResult(task_id="billing", **SAMPLE["billing"]),
           "policy": WorkerResult(task_id="policy", status="uncertain", uncertainty=["추가 약정을 찾지 못했습니다."]),
           "outage": WorkerResult(task_id="outage", status="failed", error="TimeoutError")}
required = ["billing", "policy", "outage", "credit"]  # credit은 결과가 아예 오지 않은 작업입니다.
info = join_results(partial, required)
answers = {"complete": (None, FinalAnswer(outcome="answered", text="(주입)", evidence_refs=["F-1"]), []),
           "uncertain": (None, FinalAnswer(outcome="insufficient", text="(주입)"), []),
           "failed": (TimeoutError("주입"), None, []), "결과 없음": (None, None, [])}
rows = []
for row in info["rows"]:
    kind = classify_failure(*answers[row["상태"]])
    rows.append({**row, "실패 종류": kind.value, "다음 행동": next_action(kind, retries_left=1, budget_left=True)})
show_table(rows)
print("ready:", info["ready"], "| failed:", info["failed"], "| uncertain:", info["uncertain"], "| missing:", info["missing"], "| 모델 호출 0회")

In [ ]:
worker_counts.clear()

def visible_join(state: JoinState) -> dict:
    results = {k: WorkerResult.model_validate(v) for k, v in state["results"].items()}
    return {"writer_input": join_results(results, list(ROLES))}

def after_visible_join(state: JoinState):
    failed = state["writer_input"]["failed"]
    return [Send("worker", {"task": f"{t}#2"}) for t in failed] if failed else END

visible = StateGraph(JoinState)
visible.add_node("worker", sample_worker)
visible.add_node("join", visible_join)
visible.add_conditional_edges(START, lambda state: [Send("worker", {"task": r}) for r in ROLES], ["worker"])
visible.add_edge("worker", "join")
visible.add_conditional_edges("join", after_visible_join, ["worker", END])
visible_end = visible.compile().invoke({"results": {}})
show_table([{"작업 키": k, "실행 횟수": n} for k, n in sorted(worker_counts.items())])
show_table(visible_end["writer_input"]["rows"])

first = {"outage": WorkerResult(task_id="outage", **SAMPLE["outage"]).model_dump()}
print("같은 값의 재전달:", sorted(merge_results(first, first)))
try:
    merge_results(first, {"outage": WorkerResult(task_id="outage", **SAMPLE["outage#2"]).model_dump()})
except ValueError as error:
    print("같은 키에 다른 결과:", type(error).__name__, "-", error)
print("모델 호출 0회")

**관찰할 것.** 재현 셀의 출력은 정상 완료와 구별되지 않습니다. 예외도 경고도 없고 작성 노드의 입력에 `outage`라는 글자조차 없습니다. 작성자는 가용률이 왜 없는지 알 수 없으니, 빠뜨린 채 답하거나 다른 값으로 채우게 됩니다. 조용한 합류가 실패 자체보다 위험한 이유가 여기 있습니다. 실패 자체는 재시도할 수 있지만 조용한 합류는 재시도해야 한다는 사실을 지웁니다.

`join_results` 표에서는 네 상태가 다른 행으로 나옵니다. `failed`와 `결과 없음`은 시스템 오류로, `uncertain`은 근거 부족으로 이어집니다. 다음 행동이 다릅니다. 이렇게 누락을 드러내는 책임은 합류 노드에 있습니다. 실행 횟수 표에서 `billing`, `policy`가 각각 1회이고 `outage`와 `outage#2`가 따로 있는지 확인하세요. 재시도는 새 키로 쓰고 `join_results`는 같은 작업의 마지막 시도를 읽으므로, 실패 기록은 남기고 상태 표는 최신으로 읽힙니다. `merge_results`는 같은 키에 다른 값이 오면 조용히 덮어쓰지 않고 멈춥니다.

한계도 있습니다. 이 구조는 합류 시점에 도착한 결과를 읽을 뿐, 동시에 도는 두 시도 중 늦게 도착한 쪽까지 판별하지는 않습니다. 첫 시도가 시간 초과로 처리된 뒤 뒤늦게 응답이 오는 시스템이라면 어떤 값을 더 기록해야 할까요? 시도 번호, 시작 시각, 작업 범위를 무엇으로 묶을지 적어 보세요.

3절의 `검증 문제` 칸에 나온 잘못된 참조도 같은 책임으로 이어집니다. `join_results`는 상태만 읽고 `validate_result`를 부르지 않습니다. 합류에서 근거까지 함께 검사한다면 그 문제를 어떤 실패 종류로 분류하고 어떤 다음 행동으로 이어야 할까요?

## 7. 승인과 멱등

**문제.** 지금까지의 실패는 다시 하면 그만이었습니다. 환불은 다릅니다. 1,200,000원을 두 번 돌려주면 다시 돌이키기 어렵습니다. T-1001의 P-9002는 100만원을 초과하므로 재무 담당자 승인이 필요합니다(`billing-policy.md`). 그 흐름은 다음 순서입니다.

처리 제안 → 정책·권한 검증 → 필요한 승인 → 쓰기 실행 → 결과 확인

모델은 **제안**만 합니다. 실행 여부는 코드 게이트와 사람이 정합니다. LangGraph의 `interrupt`는 노드 실행을 멈추고 값을 밖으로 보내며, `Command(resume=...)`로 재개합니다. 이때 재개가 어떻게 동작하는지가 결제 실행의 안전을 좌우합니다.

**먼저 예측하세요.** 노드 안에서 `interrupt` 앞에 원장 호출이 있습니다. 재개하면 노드의 어디부터 다시 실행될까요? `interrupt` 바로 다음 줄일까요, 노드의 첫 줄일까요? 재개까지 끝난 뒤 원장에는 몇 건이 남을까요? 답을 적은 뒤 실행하세요.

In [ ]:
# 실패 재현: 원장 호출을 interrupt 앞에 둔 노드를 재개합니다.
bad_ledger, bad_entries = PaymentLedger(), Counter()

class BadState(TypedDict, total=False):
    request: str
    answer: dict

def refund_then_ask(state: BadState) -> dict:
    bad_entries["노드 첫 줄"] += 1
    bad_ledger.refund("P-9002", 1_200_000)  # 부작용이 interrupt 앞에 있습니다. 멱등 키도 없습니다.
    answer = interrupt({"question": "P-9002 1,200,000원 환불을 승인합니까?"})
    bad_entries["interrupt 뒤"] += 1
    return {"answer": answer}

bad_graph = StateGraph(BadState)
bad_graph.add_node("refund_then_ask", refund_then_ask)
bad_graph.add_edge(START, "refund_then_ask")
bad_graph.add_edge("refund_then_ask", END)
bad_app, bad_config = bad_graph.compile(checkpointer=MemorySaver()), {"configurable": {"thread_id": "bad-1"}}
rows = []
bad_app.invoke({"request": "T-1001"}, config=bad_config)
rows.append({"시점": "승인 대기", "노드 첫 줄": bad_entries["노드 첫 줄"], "interrupt 뒤": bad_entries["interrupt 뒤"], "원장 건수": len(bad_ledger.refunds)})
bad_app.invoke(Command(resume={"approved": True}), config=bad_config)
rows.append({"시점": "승인 재개 후", "노드 첫 줄": bad_entries["노드 첫 줄"], "interrupt 뒤": bad_entries["interrupt 뒤"], "원장 건수": len(bad_ledger.refunds)})
show_table(rows)
show_table(bad_ledger.refunds)
print("승인 전에 이미 돈이 나갔고, 재개가 한 번 더 내보냈습니다. 모델 호출 0회")

In [ ]:
class ApprovalState(TypedDict, total=False):
    question: str
    proposal: dict
    decision: dict
    approved: bool
    execution: dict
    verified: dict
    path: list[str]

def approval_nodes(desk: Desk, propose, request_id: str, lose_response: bool) -> tuple[dict, Counter]:
    entries = Counter()
    def step(state, name):
        entries[name] += 1
        return [*state.get("path", []), name]
    def node_propose(state):  # 제안이 이미 있으면 모델을 부르지 않습니다.
        if state.get("proposal"):
            return {"path": step(state, "propose(입력 사용)")}
        proposal = propose(state["question"])
        return {"proposal": proposal.model_dump(), "path": step(state, "propose")}
    def gate(state):  # 검사만 합니다. 멱등 키를 코드가 정합니다.
        proposal = Proposal.model_validate(state["proposal"])
        proposal = proposal.model_copy(update={"idempotency_key": f"{request_id}:{proposal.payment_id}"})
        decision = check_refund(proposal, desk.payments(), desk.scope.as_of)
        return {"proposal": proposal.model_dump(), "decision": decision.model_dump(), "path": step(state, f"gate:{decision.decision}")}
    def approve(state):  # interrupt 앞에 부작용이 없습니다. 재개하면 이 노드가 처음부터 다시 돕니다.
        path = step(state, "approve")
        answer = interrupt({"proposal": state["proposal"], "reasons": state["decision"]["reasons"]})
        if "amount" in answer:  # 사람이 금액을 고치면 게이트를 다시 지납니다.
            return {"proposal": {**state["proposal"], "amount": answer["amount"]}, "path": path}
        return {"approved": bool(answer.get("approved")), "path": path}
    def execute(state):
        proposal, decision = Proposal.model_validate(state["proposal"]), GateDecision.model_validate(state["decision"])
        try:
            execution = execute_approved(proposal, decision, desk.ledger, approved=bool(state.get("approved")), lose_response=lose_response)
        except ResponseLost as error:
            execution = {"executed": "응답 유실", "error": str(error)}
        return {"execution": execution, "path": step(state, "execute")}
    def verify(state):  # 실행의 응답이 아니라 외부 원장을 다시 읽어 확인합니다.
        record = desk.ledger.find(state["proposal"]["idempotency_key"])
        return {"verified": {"원장에 있음": record is not None, "record": record}, "path": step(state, "verify")}
    return {"propose": node_propose, "gate": gate, "approve": approve, "execute": execute, "verify": verify}, entries

print("상태 칸:", list(ApprovalState.__annotations__), "| 모델 호출 0회")

In [ ]:
def after_gate(state):
    return {"allow": "execute", "needs_approval": "approve"}.get(state["decision"]["decision"], END)

def after_approve(state):
    if "approved" not in state:  # 사람이 금액을 고쳤습니다.
        return "gate"
    return "execute" if state["approved"] else END

def build_approval(desk: Desk, propose=None, *, request_id: str = "T-1001", lose_response: bool = False, with_verify: bool = True):
    nodes, entries = approval_nodes(desk, propose, request_id, lose_response)
    graph = StateGraph(ApprovalState)
    for name, node in nodes.items():
        if name != "verify" or with_verify:
            graph.add_node(name, node)
    graph.add_edge(START, "propose")
    graph.add_edge("propose", "gate")
    graph.add_conditional_edges("gate", after_gate, ["approve", "execute", END])
    graph.add_conditional_edges("approve", after_approve, ["gate", "execute", END])
    graph.add_edge("execute", "verify" if with_verify else END)
    if with_verify:
        graph.add_edge("verify", END)
    return graph.compile(checkpointer=MemorySaver()), entries

show_graph(build_approval(make_desk(case, label="shape"))[0])
print("모델 호출 0회. 그래프를 만드는 것은 호출이 아닙니다.")

In [ ]:
announce("모델이 T-1001 환불 제안을 작성", model_calls=5)
approval_desk, propose_meter = make_desk(case, label="approval"), ModelMeter()
PROPOSE_PROMPT = (f"환불 제안을 작성합니다. {scope_line(approval_desk)} 같은 청구서에 승인된 결제가 두 건 이상이면 중복 결제입니다. "
                  "먼저 승인된 결제는 유효 결제이고 나중 결제가 환불 대상입니다. 기준일 기준 결제일부터 90일 이내의 중복 결제만 이번 환불 대상입니다. "
                  "결제를 조회한 뒤 한 건을 골라 action=refund, payment_id, 결제 금액 amount, reason을 쓰세요. 직접 실행하지 않습니다.")

def model_propose(question: str) -> Proposal:
    agent = make_worker("billing", approval_desk, model, prompt=PROPOSE_PROMPT, response_format=Proposal)
    state = agent.invoke({"messages": [{"role": "user", "content": question}]}, config={"callbacks": [propose_meter], "recursion_limit": 40})
    return state.get("structured_response") or Proposal(reason="구조화 응답 없음")

approval_app, approval_entries = build_approval(approval_desk, model_propose)
approval_config = {"configurable": {"thread_id": "t-1001"}}
waiting = approval_app.invoke({"question": case["question"]}, config=approval_config)
ledger_before = len(approval_desk.ledger.refunds)
print("모델 호출:", propose_meter.totals()["model_calls"], "| 경로:", waiting["path"])
print("제안:", json.dumps(waiting["proposal"], ensure_ascii=False))
print("게이트:", waiting["decision"])
print("멈춘 곳:", approval_app.get_state(approval_config).next, "| interrupt 값:", [i.value["reasons"] for i in waiting.get("__interrupt__", [])])
print("승인 전 원장 건수:", ledger_before)

In [ ]:
if approval_app.get_state(approval_config).next:
    approved_end = approval_app.invoke(Command(resume={"approved": True}), config=approval_config)
    ledger_after = len(approval_desk.ledger.refunds)
    print("경로:", approved_end["path"])
    print("실행 결과:", approved_end["execution"])
    print("결과 확인:", approved_end["verified"])
    show_table([{"노드": name, "실행 횟수": count} for name, count in approval_entries.items()])
else:
    ledger_after = len(approval_desk.ledger.refunds)
    print("이번 제안은 승인 단계까지 오지 않았습니다. 게이트 결정:", waiting["decision"])
print("승인 전 원장:", ledger_before, "건 | 승인 후 원장:", ledger_after, "건 | 모델 호출 0회")

In [ ]:
FIXED_PROPOSAL = Proposal(action="refund", payment_id="P-9002", amount=1_200_000, reason="T-1001 고정 입력")

def resume_with(thread: str, answer: dict) -> dict:
    # 새 thread와 새 Desk에서 고정 제안으로 시작합니다. 모델을 부르지 않습니다.
    desk = make_desk(case, label=thread)
    app, _ = build_approval(desk)
    config = {"configurable": {"thread_id": thread}}
    app.invoke({"proposal": FIXED_PROPOSAL.model_dump()}, config=config)
    end = app.invoke(Command(resume=answer), config=config)
    return {"재개 값": str(answer), "경로": " → ".join(end["path"]), "게이트 결정": end["decision"]["decision"],
            "게이트 이유": end["decision"]["reasons"][0], "대기 중": bool(app.get_state(config).next), "원장 건수": len(desk.ledger.refunds)}

show_table([resume_with("rejected", {"approved": False}), resume_with("edited", {"approved": True, "amount": 1_000_000})])
print("금액을 고친 재개는 승인으로 바로 가지 않고 게이트를 다시 지납니다. 모델 호출 0회")

In [ ]:
# 실패 재현: 응답이 유실된 뒤 키 없이 재시도하면 두 번 환불되고, 키로 재시도하면 한 번입니다.
def retry_after_lost(key: str) -> dict:
    ledger = PaymentLedger()
    proposal = FIXED_PROPOSAL.model_copy(update={"idempotency_key": key})
    decision = check_refund(proposal, case_desk.payments(), case_desk.scope.as_of)
    try:
        execute_approved(proposal, decision, ledger, approved=True, lose_response=True)
    except ResponseLost as error:
        lost = str(error)
    after_loss = len(ledger.refunds)
    retried = execute_approved(proposal, decision, ledger, approved=True)
    return {"멱등 키": key or "(없음)", "유실 메시지": lost, "유실 직후 원장": after_loss, "재시도 후 원장": len(ledger.refunds),
            "재시도 응답 refund_id": retried["record"]["refund_id"], "원장의 refund_id": [r["refund_id"] for r in ledger.refunds]}

lost_rows = [retry_after_lost(""), retry_after_lost("T-1001:P-9002")]
show_table(lost_rows)
print("키 없이 재시도:", lost_rows[0]["재시도 후 원장"], "건 | 키로 재시도:", lost_rows[1]["재시도 후 원장"], "건 | 모델 호출 0회")

In [ ]:
def recover(proposal: Proposal, decision: GateDecision, ledger: PaymentLedger, first_lost: bool) -> list[dict]:
    # 응답이 없으면 먼저 외부 결과를 조회합니다. 있으면 재시도하지 않고, 없을 때만 다시 실행합니다.
    steps = []
    try:
        record = execute_approved(proposal, decision, ledger, approved=True, lose_response=first_lost)["record"]
        steps.append({"단계": "실행", "결과": "응답 받음", "refund_id": record["refund_id"]})
    except ResponseLost as error:
        steps.append({"단계": "실행", "결과": f"응답 유실: {error}", "refund_id": ""})
        found = ledger.find(proposal.idempotency_key)
        steps.append({"단계": "ledger.find", "결과": "기록 있음. 재시도하지 않습니다." if found else "기록 없음. 다시 실행합니다.", "refund_id": found["refund_id"] if found else ""})
    steps.append({"단계": "원장 건수", "결과": len(ledger.refunds), "refund_id": ""})
    return steps

keyed = FIXED_PROPOSAL.model_copy(update={"idempotency_key": "T-1001:P-9002"})
keyed_decision = check_refund(keyed, case_desk.payments(), case_desk.scope.as_of)
show_table(recover(keyed, keyed_decision, PaymentLedger(), first_lost=True))
print("모델 호출 0회")

**관찰할 것.** 재현 셀의 표에서 `노드 첫 줄` 칸과 `interrupt 뒤` 칸을 비교하세요. 재개는 멈춘 줄에서 이어지는 것이 아니라 노드를 처음부터 다시 실행합니다. 그래서 승인 대기 시점에 이미 원장에 1건이 있고, 재개가 끝나면 2건이 됩니다. 사람이 승인하기도 전에 돈이 나갔다는 점이 두 번 나간 것만큼 심각합니다. 올바른 그래프의 `실행 횟수` 표에서 `approve`가 `execute`보다 많은 것을 찾으세요. `approve` 노드도 재개 때 처음부터 다시 돕니다. 그 앞에 부작용이 없어 해가 없을 뿐입니다.

제안 셀은 모델 결과에 따라 달라집니다. `gate:needs_approval`로 멈췄다면 `멈춘 곳`이 `approve`이고 원장은 0건이어야 합니다. 모델이 다른 결제나 다른 금액을 제안했다면 게이트가 거절해 멈추지 않습니다. 그때는 승인 칸이 비고, 거절 이유가 곧 관찰 결과입니다. 어느 쪽이든 모델의 제안만으로 원장이 바뀌는 경로는 없어야 합니다. 거절 행과 금액 수정 행의 `원장 건수`, 그리고 금액 수정 행의 `게이트 이유`를 확인하세요. 금액을 고치면 승인했더라도 게이트를 다시 지납니다. 이것이 승인의 전부는 아닙니다. 승인한 내용과 실제 실행 인자를 묶는 승인 바인딩은 Day08에서 다룹니다.

응답 유실 표는 `재시도 후 원장` 칸을 읽습니다. 원장은 기록을 남긴 뒤 응답만 잃었습니다. 키 없는 재시도는 원장이 같은 요청인지 알 수 없어 새 환불을 만들고, 키가 있으면 같은 `refund_id`를 돌려줍니다. 그다음 셀은 더 나은 순서입니다. 응답이 없으면 곧바로 다시 부르지 않고 `ledger.find`로 외부 결과부터 조회합니다.

체크포인트는 우리 그래프의 진행을 저장할 뿐입니다. `MemorySaver`가 있어도 첫 번째 재현에서 두 건이 남았습니다. `idempotency_key`는 외부 시스템이 지원하고 강제해야 뜻이 있으며, 재개한다고 exactly-once가 자동으로 보장되지 않습니다. 여기의 `PaymentLedger`는 그 강제를 흉내 낸 가짜입니다. 실제 결제 API가 같은 키 처리, 같은 보관 기간, 같은 조회 기능을 제공한다고 가정하면 안 됩니다. 실제 연동 전에 그 API의 문서로 확인해야 합니다.

## 8. 완성형 조립: Workflow 안에 필요한 Agent만 배치합니다

**문제.** 앞 절의 부품을 한 그래프에 모으면 어떤 모양이 될까요? 전체를 Agent 하나에게 맡기지도, 전체를 고정 코드로 짜지도 않습니다. 코드가 경로와 쓰기를 정하고, 판단이 필요한 노드에서만 모델과 Agent를 씁니다. 라우팅, 조사, 작성은 모델이 맡습니다. 범위 확인, 합류, 검증, 게이트, 승인, 실행은 코드가 맡습니다.

소스를 읽으며 다음을 찾아보세요. `gate` 노드에 왜 부작용이 없을까요? `after_join`은 어떤 조건에서 `retry`로 갈까요? `execute`는 어떤 함수를 통해 원장에 닿을까요? 아래 표는 각 노드를 1절의 층에 대응시킵니다. 그림의 점선은 조건부 간선인데 어떤 조건에서 어느 갈래로 가는지는 그림에 적혀 있지 않으므로, 갈림 조건은 아래 셀이 출력하는 `build_hybrid` 소스에서 찾거나 그림 아래 간선 표의 `조건부` 열이 `True`인 행을 찾아 그 간선을 고르는 라우팅 함수에서 읽어야 합니다.

In [ ]:
print(inspect.getsource(build_hybrid))
hybrid_shape = build_hybrid(make_desk(case, label="shape"), model)  # 만들기만 합니다. 호출이 아닙니다.
show_graph(hybrid_shape)
NODE_LAYERS = {"scope": ("Harness", "코드", "범위 밖 고객을 멈춥니다"), "router": ("Prompt, Graph", "모델", "경로와 조사 범위를 고릅니다"),
               "fast": ("Prompt, Context", "Agent", "단순 질문을 바로 답합니다"), "worker": ("Context", "Agent", "WorkerTask를 받아 WorkerResult를 돌려줍니다"),
               "join": ("Graph", "코드", "결과를 합치고 상태를 읽습니다"), "retry": ("Loop", "Agent", "실패한 작업만 새 키로 다시 돌립니다"),
               "writer": ("Context", "Agent", "결과와 calculate_credit으로 답을 씁니다"), "validate": ("Harness, Loop", "코드", "근거와 숫자를 검사하고 재작성합니다"),
               "gate": ("Harness", "코드", "게이트 판단으로 경로를 나눕니다"), "approve": ("Harness", "사람", "interrupt로 승인을 기다립니다"), "execute": ("Harness", "코드", "gate_answer로만 원장에 닿습니다")}
show_table([{"노드": node, "층": layer, "누가 하는가": actor, "역할": role} for node, (layer, actor, role) in NODE_LAYERS.items()])
print("모델 호출 0회")

In [ ]:
announce("완성형 조립으로 T-1001 처리", model_calls=16)
hybrid_row = run_case("hybrid", case, model=model)  # approval이 None이면 승인 대기로 끝납니다.
print("status:", hybrid_row["status"], "| outcome:", hybrid_row["outcome"], "| 모델 호출:", hybrid_row["model_calls"], "| 시간:", hybrid_row["elapsed_seconds"], "초")
print(hybrid_row["answer"])
print("불확실성:", hybrid_row.get("uncertainty"))

In [ ]:
show_table([{k: hybrid_row[k] for k in ("structure", "status", "outcome", "control_owner", "final_speaker", "model_calls", "tool_calls", "input_tokens", "output_tokens",
                                        "context_bytes", "max_worker_overlap_seconds", "ledger_refunds", "policy_violations")}])
print(timeline(hybrid_row["spans"]))
hybrid_check = evaluate_row(hybrid_row, gold)
show_table([{k: hybrid_check[k] for k in ("자동 검사", "기대 outcome", "실제 outcome", "누락 사실", "금지 포함", "누락 문서", "원장 건수", "실패 칸")}])
print("제안:", hybrid_row.get("proposal"))
print("인용 문서:", hybrid_row["cited_docs"], "| 도구 호출:", Counter(hybrid_row["tools_called"]))
print("검증 사건:", [e["problems"] for e in hybrid_row["events"] if e["kind"] == "validate"])
print("오류:", hybrid_row["errors"] or "없음", "| usage 누락:", hybrid_row["usage_missing"], "| 이 셀의 모델 호출 0회")

**관찰할 것.** 처음의 업무 목표를 끝까지 추적하세요. 고객의 요청은 두 가지였습니다. 중복 결제의 취소와 보상안 확인입니다. `answer`에 환불 대상과 금액, 가용률, 크레딧 비율과 금액, 신청 기한이 있는지 읽고 `자동 검사` 칸과 비교합니다. 실패 칸이 있으면 어느 층의 문제인지 1절 표로 돌아가 말해 보세요. 필수 사실이 없다면 Context의 문제일 수 있고, 잘못된 금액이 있다면 도구를 쓰지 않은 Prompt나 Harness의 문제일 수 있습니다. 자동 검사가 통과했더라도 원문과 대조하는 일은 남습니다. 반대로 `실패 칸`이 `필수 사실` 하나이고 `누락 사실`이 `9월20일`이라면, 먼저 답에 `2026-09-20`처럼 다른 표기가 있는지 찾으세요. 있다면 답이 아니라 자동 검사의 한계입니다.

`timeline`에 `writer`가 두 줄이면 재작성이 일어난 것입니다. `검증 사건` 줄의 첫 목록이 비어 있지 않으면 `validate` 노드가 문제를 찾아 작성 노드로 돌려보낸 것이고, 둘째 목록이 비어 있으면 재작성이 문제를 해소한 것입니다. 이 재작성에 쓴 호출도 `model_calls`에 들어갑니다. 재작성 없이 끝났다면 목록이 하나입니다.

실행과 미실행의 결과까지 설명할 수 있어야 합니다. `outcome`이 `needs_approval`이고 `ledger_refunds`가 0이면 환불은 아직 일어나지 않았습니다. 답의 문장이 "환불했습니다"처럼 읽힌다면 `policy_violations` 칸과 함께 살펴보세요. 이 칸은 outcome과 원장을 대조할 뿐, 문장의 시제까지 보지는 않습니다. `timeline`에서 Worker 구간이 세로로 겹치는지도 보세요. 이 값 역시 1회 관찰입니다. 05에서 같은 조건으로 구조를 비교할 때 다시 쓰게 됩니다.

## 직접 해보기

네 칸 모두 조건 하나만 바꿉니다. 먼저 무엇이 달라질지 적은 뒤 돌리세요. 해보기 1만 모델을 부르므로 `RUN_EXERCISE`를 켜야 합니다. 나머지 셋은 모델을 부르지 않고 바로 실행됩니다.

### 해보기 1

작업 계약의 한 줄(한계 또는 권한)을 지우고 2절의 계약 Worker를 다시 돌립니다. 달라지는 칸을 찾으세요. 달라지지 않았다면 1회 관찰로 무엇을 말할 수 있는지도 적습니다.

In [ ]:
RUN_EXERCISE = False
REMOVE_LINE = "한계:"  # "권한:"으로 바꿔 볼 수 있습니다.
if RUN_EXERCISE:
    announce("해보기 1: 계약 한 줄 지우기", model_calls=4)
    contract = "\n".join(line for line in TASK_CONTRACT.splitlines() if not line.startswith(REMOVE_LINE))
    exercise_desk = make_desk(case, label="exercise-1")
    content = contract.format(goal=GOAL, entity=exercise_desk.scope.entity, as_of=exercise_desk.scope.as_of.isoformat(), facts="없음")
    state = make_worker("outage", exercise_desk, model).invoke({"messages": [{"role": "user", "content": content}]}, config={"recursion_limit": 40})
    answer = state["structured_response"]
    print(content)
    show_table([{"지시": name, "status": r.status, "사실 수": len(r.facts), "근거 수": len(r.evidence_refs), "불확실성 수": len(r.uncertainty),
                 "숫자의 출처": source_of_numbers(" ".join([*r.facts, *r.uncertainty]), d)}
                for name, r, d in (("전체 계약", contract_result, contract_desk), (f"{REMOVE_LINE} 삭제", answer, exercise_desk))])
else:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")

### 해보기 2

`trim_payload`는 원문을 먼저, 사실을 나중에 뺍니다. 아래 `DROP_ORDER`를 바꿔 어떤 항목이 먼저 사라지게 할지 정하세요. `uncertainty`를 먼저 지우면 작성자가 무엇을 모른 채 답하게 될까요? 그 선택이 어떤 답을 위험하게 만드는지 적습니다.

In [ ]:
DROP_ORDER = ["uncertainty", "facts", "quotes"]  # 먼저 버릴 칸부터 적습니다.
MAX_BYTES = int(full_size * 0.5)

def my_trim(result: WorkerResult, quotes: dict[str, str], max_bytes: int, order: list[str]) -> dict:
    payload = {"facts": list(result.facts), "evidence_refs": list(result.evidence_refs), "uncertainty": list(result.uncertainty), "quotes": dict(quotes)}
    size = lambda: len(json.dumps(payload, ensure_ascii=False).encode("utf-8"))
    dropped = []
    for field in order:
        while size() > max_bytes and payload[field]:
            item = payload[field].popitem()[0] if isinstance(payload[field], dict) else payload[field].pop()
            dropped.append(f"{field}: {item[:20]}")
    return {"payload": payload, "bytes": size(), "dropped": dropped}

mine, standard = my_trim(policy_result, policy_quotes, MAX_BYTES, DROP_ORDER), trim_payload(policy_result, policy_quotes, MAX_BYTES)
show_table([{"순서": name, "바이트": t["bytes"], "남은 사실": len(t["payload"]["facts"]), "남은 원문": len(t["payload"]["quotes"]),
             "남은 불확실성": len(t["payload"]["uncertainty"]), "뺀 항목 수": len(t["dropped"])}
            for name, t in (("내 순서", mine), ("trim_payload", standard))])
print("내 순서로 뺀 항목:", mine["dropped"], "| 모델 호출 0회")

### 해보기 3

`next_action`을 고쳐 정책 불가도 재조사하게 합니다. 5절의 주입 그래프에서 경로가 어떻게 바뀌는지 보세요. 재조사를 세 번 허용하면 무엇이 늘어나고, 그것이 고객에게 어떤 뜻인지 말해 보세요. `정책 불가` 행의 `시도 수`가 달라지는데, 두 번째 시도에서 통과로 끝났다면 그 통과를 믿어도 될까요? 주입 대본이 그렇게 짜여 있기 때문인지, 재조사가 규정을 바꿀 수 있어서인지 구분해 답하세요.

In [ ]:
RETRIES = 3

def reinvestigate_policy(kind: FailureKind, *, retries_left: int, budget_left: bool) -> str:
    if kind is FailureKind.POLICY_DENIED:
        return "reinvestigate" if retries_left > 0 and budget_left else "stop_with_reason"
    return next_action(kind, retries_left=retries_left, budget_left=budget_left)

SCENARIOS["정책 불가(계속 거절)"] = ["denied"]
show_table(run_scenarios(reinvestigate_policy, RETRIES))
show_table(run_scenarios(next_action, RETRIES))
del SCENARIOS["정책 불가(계속 거절)"]
print("모델 호출 0회")

### 해보기 4

승인 그래프에서 `verify`를 빼고 응답 유실을 넣습니다. 두 행의 `실행 결과`와 `결과 확인`, `원장 건수`를 비교하세요. `verify`가 없으면 돈이 나갔는지 그래프 상태만으로 알 수 있습니까? 이때 사람이 "실패했으니 다시 실행"을 누르면 어떤 일이 생길까요?

In [ ]:
WITH_VERIFY = [True, False]
rows = []
for with_verify in WITH_VERIFY:
    desk = make_desk(case, label="exercise-4")
    app, _ = build_approval(desk, lose_response=True, with_verify=with_verify)
    config = {"configurable": {"thread_id": f"lost-{with_verify}"}}
    app.invoke({"proposal": FIXED_PROPOSAL.model_dump()}, config=config)
    end = app.invoke(Command(resume={"approved": True}), config=config)
    rows.append({"verify": with_verify, "경로": " → ".join(end["path"]), "실행 결과": end["execution"].get("executed"),
                 "결과 확인": end.get("verified", {}).get("원장에 있음", "알 수 없음"), "원장 건수": len(desk.ledger.refunds)})
show_table(rows)
print("모델 호출 0회")

In [ ]:
# 아래 답을 직접 바꿔 쓰세요. 06이 이 기록을 모아 읽습니다.
ANSWER = ("무엇을 보고 무엇을 실행할 수 있는가: (내 업무의 Worker마다 볼 수 있는 자료와 쓸 수 있는 도구, 실행은 누가 하는가)\n"
          "언제 재시도하고 언제 전환하는가: (어떤 실패에 재시도하고, 어떤 실패에 사람에게 넘기거나 멈추는가)\n"
          "예산과 종료: (한 실행의 모델 호출 상한, 그 근거, 넘었을 때 무엇을 남기는가)")
path = write_design_log("Context·권한과 예산·종료: 무엇을 보고 무엇을 실행하며 언제 멈출까?", ANSWER)
print("설계 기록을 썼습니다:", path.relative_to(Path.cwd().resolve()))

## 정리

이 노트북에서 그래프에 넣은 것을 층별로 다시 적습니다.

- **Prompt.** 역할 이름이 아니라 작업, 근거, 출력, 한계, 권한을 적은 계약으로 Worker를 정의했습니다. 계약은 평균 품질이 아니라 다음 단계가 읽을 칸을 정합니다.
- **Context.** 사실, 근거, 불확실성을 칸으로 나눠 전달했습니다. 스키마 통과와 근거 검증은 다른 검사이며, 그 둘을 지나도 뜻이 맞다는 보장은 없습니다. 줄이는 일은 항목 단위로 하고, 뺀 것을 기록합니다.
- **Harness.** 예산을 모델 밖에서 하나로 세고, 넘으면 멈춰서 드러냈습니다. 쓰기는 코드 게이트를 지나야만 원장에 닿습니다.
- **Graph.** 합류 노드가 필수 작업별 상태를 드러냅니다. 조용히 빠뜨리지 않습니다.
- **Loop.** 실패의 종류가 다음 행동을 정합니다. 재시도, 재조사, 멈춤, 사람에게 넘기기는 서로 대체할 수 없습니다.
- **승인과 멱등.** 부작용은 `interrupt` 뒤의 별도 노드에 둡니다. 재개는 노드를 처음부터 다시 돌립니다. 체크포인트는 그래프의 진행만 저장하고, 중복 실행을 막는 일은 외부 시스템의 멱등 키에 달려 있습니다.

**Day08에서 이어지는 항목.** 승인 기록과 실행 인자를 묶는 승인 바인딩(도구, 인자 해시, 만료)과 재검증, 입력과 출력의 Guardrail, 권한과 메모리 방어를 다룹니다.

**읽을거리.** 이 과정은 1절에서 적은 정의를 씁니다. "Graph Engineering"은 2026년에 퍼진 외부 용어이고 출처마다 범위가 다릅니다. 관심이 있으면 다음 두 글을 비교용으로 읽어 보세요. 어느 쪽 정의가 옳은지보다, 두 글이 각각 무엇을 그래프의 일로 보는지를 위 층 지도에 대응시켜 보는 것을 권합니다.

- https://towardsdatascience.com/graph-engineering-for-ai-agents-from-prompts-and-loops-to-workflows/
- https://rickhigh.substack.com/p/graph-engineering-is-two-ideas-wearing

## 확인 질문

1. 인용 ID가 있고 스키마를 통과한 `WorkerResult`도 왜 신뢰할 수 없습니까? 3절의 잘못된 결과 표에서 어느 행이 근거입니까?
2. 3절의 전달물 비교가 전달 형태의 효과를 보이기 위해 Worker를 다시 돌리지 않은 이유는 무엇입니까? 다시 돌렸다면 무엇과 섞입니까?
3. 시스템 오류와 근거 부족의 다음 행동은 왜 달라야 합니까?
4. 실패한 Worker를 조용히 빠뜨린 합류가 왜 그 실패 자체보다 위험합니까?
5. `MemorySaver`가 있는데 왜 `idempotency_key`가 필요합니까? 7절의 첫 번째 재현과 응답 유실 재현의 차이는 무엇입니까?
6. 여기 원장의 멱등 보장을 실제 결제 API에 그대로 기대할 수 없는 이유는 무엇입니까?

다음 05에서는 이 노트북의 부품을 고정한 채 구조만 바꿉니다. 실패 분류에서 출발해 같은 조건에서 구조를 재고, 나누는 이유와 나누지 않는 이유를 숫자로 뒷받침합니다.